<a href="https://colab.research.google.com/github/angelosbc/analise-de-sentimento-steam/blob/main/Steam_sentimento_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. Inserção do Dataset via Kaggle Hub**

In [1]:
import kagglehub
import os
import pandas as pd

# Download do dataset pré-processado do meu repositório público

path = kagglehub.dataset_download("angelosbc/steam-reviews-in-portuguese-pt-br-2021")
print("Pasta do dataset:", path)

# Carrega o CSV filtrado no Pandas
caminho_csv = os.path.join(path, "steam_reviews_brazilian.csv")
df = pd.read_csv(caminho_csv)

# 3. Mostra total de avaliações e alguns dados
print(f"Total de avaliações em português: {len(df):,}")
df.head()

100%|██████████| 96.6M/96.6M [00:03<00:00, 32.1MB/s]

Extracting files...


Pasta do dataset: /root/.cache/kagglehub/datasets/angelosbc/steam-reviews-in-portuguese-pt-br-2021/versions/1
Total de avaliações em português: 918,910


,Unnamed: 0,app_id,app_name,review_id,language,review,timestamp_created,timestamp_updated,recommended,votes_helpful,...,steam_purchase,received_for_free,written_during_early_access,author.steamid,author.num_games_owned,author.num_reviews,author.playtime_forever,author.playtime_last_two_weeks,author.playtime_at_review,author.last_played
0,29,292030,The Witcher 3: Wild Hunt,85177505,brazilian,Se um dia alguém falar que esse jogo é ruim na...,1611368498,1611368498,True,1,...,True,False,False,76561198844659805,70,4,11115.0,2252.0,11115.0,1.611186e+09
1,30,292030,The Witcher 3: Wild Hunt,85176839,portuguese,bom demais\n,1611367482,1611367482,True,0,...,True,False,False,76561198847379347,4,1,555.0,465.0,555.0,1.611367e+09
2,32,292030,The Witcher 3: Wild Hunt,85176661,brazilian,NaN,1611367193,1611367193,True,0,...,True,False,False,76561198076880796,127,13,875.0,752.0,826.0,1.611370e+09
3,34,292030,The Witcher 3: Wild Hunt,85176249,brazilian,Obra prima!!!,1611366524,1611366524,True,0,...,True,False,False,76561198957873353,32,1,2888.0,1475.0,2888.0,1.611366e+09
4,43,292030,The Witcher 3: Wild Hunt,85173023,brazilian,Jogão da porra.,1611361229,1611361229,True,0,...,True,False,False,76561198141110905,59,4,20193.0,3692.0,20193.0,1.611297e+09


**2. Pré-processamento Textual e Divisão dos Dados**

In [2]:
import re
from sklearn.model_selection import train_test_split

# 1. Remove linhas nulas nas colunas essenciais
print("-> Removendo valores ausentes...")
df = df.dropna(subset=['review', 'recommended'])

# 2. Converte a coluna alvo para inteiro (0 = Negativo, 1 = Positivo)
df['label'] = df['recommended'].astype(int)

# 3. Função de normalização e limpeza sintática do texto
def limpar_texto(texto):
    if not isinstance(texto, str):
        return ""
    # Converte para minúsculas
    texto = texto.lower()
    # Remove URLs completas
    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto, flags=re.MULTILINE)
    # Remove tags HTML
    texto = re.sub(r'<.*?>', '', texto)
    # Mantém apenas letras acentuadas e espaços, removendo pontuações/símbolos
    texto = re.sub(r'[^a-záàâãéèêíïóôõöúçñ\s]', ' ', texto)
    # Remove espaços em branco redundantes
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

# 4. Aplica a limpeza e filtra textos com menos de 4 caracteres
print("-> Aplicando rotina de limpeza no texto...")
df['clean_review'] = df['review'].apply(limpar_texto)
df = df[df['clean_review'].str.len() >= 4]

print(f"Total de registros válidos pós-limpeza: {len(df):,}")

# 5. Divisão estratificada (70% Treino, 15% Validação, 15% Teste)
print("-> Realizando divisão estratificada (70/15/15)...")
X = df['clean_review']
y = df['label']

# Primeiro corte: separa 70% treino e 30% temporário
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Segundo corte: divide os 30% temporários igualmente entre validação e teste
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Conjunto de Treino:     {len(X_train):,} amostras")
print(f"Conjunto de Validação:  {len(X_val):,} amostras")
print(f"Conjunto de Teste:      {len(X_test):,} amostras")

-> Removendo valores ausentes...
-> Aplicando rotina de limpeza no texto...
Total de registros válidos pós-limpeza: 853,982
-> Realizando divisão estratificada (70/15/15)...
Conjunto de Treino:     597,787 amostras
Conjunto de Validação:  128,097 amostras
Conjunto de Teste:      128,098 amostras


**3. Baseline 1 (TF-IDF + Regressão Logística)**

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, roc_auc_score, accuracy_score

# 1. Vetorização por TF-IDF (Unigramas + Bigramas limitados a 5.000 termos)
print("-> Ajustando o vetorizador TF-IDF no conjunto de treino...")
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)

# Transforma os conjuntos de dados em matrizes esparsas
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 2. Inicialização e treinamento da Regressão Logística com penalização L2
print("-> Treinando o classificador de Regressão Logística...")
modelo_lr = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
modelo_lr.fit(X_train_vec, y_train)

# 3. Inferência sobre o conjunto de teste
print("-> Gerando predições e probabilidades para o teste...")
y_pred = modelo_lr.predict(X_test_vec)
y_prob = modelo_lr.predict_proba(X_test_vec)[:, 1]

# 4. Exibição das métricas consolidadas
print("\n" + "="*45)
print("   RESULTADOS DO BASELINE 1 (TF-IDF + LR)   ")
print("="*45)
print(f"Acurácia:        {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-Score Macro:  {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"ROC-AUC:         {roc_auc_score(y_test, y_prob):.4f}")
print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_test, y_pred, target_names=['Não Recomenda (0)', 'Recomenda (1)']))

-> Ajustando o vetorizador TF-IDF no conjunto de treino...
-> Treinando o classificador de Regressão Logística...
-> Gerando predições e probabilidades para o teste...

   RESULTADOS DO BASELINE 1 (TF-IDF + LR)   
Acurácia:        0.9618
F1-Score Macro:  0.7899
ROC-AUC:         0.9548

Relatório de Classificação Detalhado:
                   precision    recall  f1-score   support

Não Recomenda (0)       0.78      0.49      0.60      7503
    Recomenda (1)       0.97      0.99      0.98    120595

         accuracy                           0.96    128098
        macro avg       0.87      0.74      0.79    128098
     weighted avg       0.96      0.96      0.96    128098

